# 004 Skills

这是 LangChain Multi-agent 学习线的第四份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent/skills

学习目标：

1. 理解 skill 是按需加载的专业能力包，不只是 prompt 模板
2. 区分 skill、tool、subagent、handoff 的边界
3. 学会用 skill registry 表达可选能力
4. 学会根据用户意图只加载相关 skill 指令和工具
5. 对比本仓库 `.agents/skills` 的设计

这一讲使用 fake model，不消耗真实模型额度。

## 1. Skill 解决什么问题

前面几讲我们已经见过：

- subagent：把任务委派给另一个 agent
- handoff：把后续控制权交给另一个 active agent / step

skill 解决的是另一个问题：

```text
当前 agent 仍然负责对话和控制流，
但它可以按需加载某个专业能力的说明、参考资料、资源和工具。
```

所以 skill 更像 Java 里的一个“可插拔业务能力模块”：

```text
Skill = 使用说明 + 触发条件 + 参考资料 + 可用工具 + 输出约定
```

它不是另一个聊天窗口，也不一定是另一个模型。

## 2. Skill、Tool、Subagent、Handoff 的边界

| 能力 | 主要解决的问题 | 谁控制下一步 | 是否需要新 agent |
| --- | --- | --- | --- |
| tool | 执行一个确定动作 | 当前 agent | 不需要 |
| skill | 按需加载一组专业上下文和工具 | 当前 agent | 不需要 |
| subagent | 隔离上下文，委派局部任务 | supervisor / coordinator | 通常需要 |
| handoff | 后续对话控制权转移 | 新 active agent / step | 通常需要 |

判断口径：

```text
只是多一个函数能力：tool
需要一套专业说明和工具组合：skill
需要隔离上下文做局部任务：subagent
需要后续都换角色接管：handoff
```

In [33]:
from dataclasses import dataclass
from typing import Any, Callable

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


def print_messages(result: dict) -> None:
    for message in result.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)

## 3. 用 SkillSpec 表达一个 skill

当前安装的 LangChain 版本没有单独的 `Skill` 类。

但这不影响我们学习核心思想：

```text
skill registry 保存能力包
router / selector 判断本轮需要哪个 skill
middleware 把对应 skill 的 instructions 和 tools 注入 agent
```

下面用普通 Python 定义一个最小 `SkillSpec`。

In [34]:
@dataclass(frozen=True)
class SkillSpec:
    name: str
    description: str
    trigger_keywords: list[str]
    instructions: str
    tools: list[Any]


@tool
def lookup_invoice_total(invoice_id: str) -> str:
    """Look up the total amount for an invoice."""
    totals = {
        "INV-001": "1280.50 CNY",
        "INV-002": "430.00 CNY",
    }
    return totals.get(invoice_id, "invoice not found")


@tool
def summarize_refund_rule(customer_tier: str) -> str:
    """Summarize the refund rule for a customer tier."""
    rules = {
        "gold": "Gold customers can request a refund within 30 days.",
        "standard": "Standard customers can request a refund within 7 days.",
    }
    return rules.get(customer_tier, "Use the standard refund rule.")


SKILL_REGISTRY = {
    "invoice-audit": SkillSpec(
        name="invoice-audit",
        description="检查发票、账单和金额相关问题。",
        trigger_keywords=["invoice", "发票", "账单", "金额"],
        instructions="你是发票审查助手。回答前优先核对 invoice_id 和金额，不要编造账单数据。",
        tools=[lookup_invoice_total],
    ),
    "refund-policy": SkillSpec(
        name="refund-policy",
        description="解释退款规则、退款周期和客户等级差异。",
        trigger_keywords=["refund", "退款", "退费", "退订"],
        instructions="你是退款规则助手。先确认 customer_tier，再解释适用规则。",
        tools=[summarize_refund_rule],
    ),
}

ALL_SKILL_TOOLS = [lookup_invoice_total, summarize_refund_rule]

## 4. 根据用户意图选择 skill

这里先用确定性关键词选择。

真实系统里可以替换成：

- 模型结构化输出
- embedding 检索
- 业务规则
- 用户当前页面 / 业务上下文

但不管怎么选，系统层都应该保留边界控制：模型不能随意加载所有 skill。

In [35]:
def select_skill(user_message: str) -> str | None:
    lowered = user_message.lower()
    for skill_name, skill in SKILL_REGISTRY.items():
        if any(keyword.lower() in lowered for keyword in skill.trigger_keywords):
            return skill_name
    return None


examples = [
    "帮我检查发票 INV-001 的金额",
    "gold 用户想退款，规则是什么？",
    "你好，今天聊点别的。",
]

for text in examples:
    print(text, "=>", select_skill(text))

帮我检查发票 INV-001 的金额 => invoice-audit
gold 用户想退款，规则是什么？ => refund-policy
你好，今天聊点别的。 => None


## 5. 用 middleware 按需加载 skill

skill 被选中后，需要把两类东西注入 agent：

1. `instructions`：本 skill 的专业规则
2. `tools`：本 skill 允许使用的工具集合

这就是 Skills 的重点：

```text
不是把所有能力永久塞进 prompt，
而是在需要时加载刚好够用的能力包。
```

In [36]:
class SkillState(AgentState):
    selected_skill: str | None


@wrap_model_call
def apply_skill_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    selected_skill = request.state.get("selected_skill")
    skill = SKILL_REGISTRY.get(selected_skill or "")

    if skill is None:
        print("loaded skill: none")
        request = request.override(
            system_prompt="你是普通客服助手。没有加载专业 skill 时，不要编造业务数据。",
            tools=[],
        )
    else:
        print("loaded skill:", skill.name)
        request = request.override(
            system_prompt=skill.instructions,
            tools=skill.tools,
        )

    return handler(request)

## 6. 跑通 invoice-audit skill

fake model 第一轮会调用 `lookup_invoice_total`。

注意：agent 本身可以注册多个工具，但 middleware 在本轮只把 `invoice-audit` 对应的工具暴露给模型。

In [37]:
invoice_message = "帮我检查发票 INV-001 的金额"
selected_skill = select_skill(invoice_message)

skill_model = ToolCallingFakeModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "lookup_invoice_total",
                    "args": {"invoice_id": "INV-001"},
                    "id": "call_1",
                }
            ],
        ),
        AIMessage(content="发票 INV-001 的金额是 1280.50 CNY。"),
    ]
)

skill_agent = create_agent(
    model=skill_model,
    tools=ALL_SKILL_TOOLS,
    state_schema=SkillState,
    middleware=[apply_skill_context],
)

skill_result = skill_agent.invoke(
    {
        "messages": [{"role": "user", "content": invoice_message}],
        "selected_skill": selected_skill,
    }
)

print_messages(skill_result)
print("selected_skill:", skill_result.get("selected_skill"))

loaded skill: invoice-audit
loaded skill: invoice-audit
human 帮我检查发票 INV-001 的金额
ai 
tool_calls: [{'name': 'lookup_invoice_total', 'args': {'invoice_id': 'INV-001'}, 'id': 'call_1', 'type': 'tool_call'}]
tool 1280.50 CNY
ai 发票 INV-001 的金额是 1280.50 CNY。
selected_skill: invoice-audit


## 7. 和本仓库 `.agents/skills` 的对应关系

本仓库已经有 repo-local Skills。

它们不是普通 Python 函数，而是目录化能力包。

In [38]:
from pathlib import Path

skill_root = Path(".agents/skills")

if skill_root.exists():
    for skill_dir in sorted(path for path in skill_root.iterdir() if path.is_dir()):
        print(skill_dir.name)
else:
    print(".agents/skills does not exist from the current working directory")

fund-query
weather-query-assistant


本仓库的 skill 结构一般是：

```text
.agents/skills/<skill-name>/
  SKILL.md              # 核心使用说明
  agents/openai.yaml    # 可选，给 agent 的元信息
  references/           # 可选，大段参考资料
  scripts/              # 可选，确定性脚本
```

这和本讲的 `SkillSpec` 是同一个思想：

| 本讲示例 | 仓库真实 Skill |
| --- | --- |
| `name` | 目录名 / manifest 名称 |
| `description` | `SKILL.md` 描述 |
| `trigger_keywords` | 触发说明 / planner hint |
| `instructions` | `SKILL.md` 主体 |
| `tools` | scripts / managed tools |

区别是：真实 Skill 会把内容放到文件系统里，方便复用、安装、版本管理和按需加载。

## 8. Skill 的设计要点

设计 skill 时要回答：

1. 这个能力什么时候应该被加载？
2. 它需要哪些最小说明？
3. 哪些资料应该放在 `references/`，而不是塞进主 prompt？
4. 它是否需要确定性脚本？
5. 它暴露的工具是否需要审批？
6. 它的输出格式是否稳定？
7. 它是否会访问用户数据、文件系统或外部服务？

核心原则：

```text
Skill 是上下文和工作流模块，
不是把一段 prompt 复制到系统提示词里。
```

## 9. 本讲练习

请判断下面能力更适合做 tool、skill、subagent 还是 handoff：

1. 查询一个订单状态，需要调用确定接口。
2. 处理基金查询，需要固定数据源、格式说明、脚本和兜底规则。
3. 独立检查一组代码改动是否符合测试结果。
4. 用户从普通客服进入售后投诉，后续都由投诉专员接管。

参考答案：

1. tool
2. skill
3. subagent
4. handoff

## 10. 本讲小结

这一讲的核心：

```text
Skill 是按需加载的专业能力包。
```

你现在应该能判断：

- skill 和 tool 的区别
- skill 和 subagent 的区别
- 为什么 skill 需要触发条件
- 为什么 skill 应该按需加载，而不是全部塞进上下文
- 本仓库 `.agents/skills` 为什么采用目录化设计

下一讲可以继续进入 Router。